# 02_contrafractual — cuello de botella dominante por distrito

Reutiliza el mismo Causal Forest de `02_causal_forest.ipynb` (se re-entrena aquí con los mismos datos y `random_state`, así que da exactamente el mismo modelo, sin depender de guardar el objeto en disco). Para cada distrito, simula "¿qué pasaría con τ si este distrito tuviera mejor acceso a X, todo lo demás constante?" — una variable a la vez — para identificar cuál es el cuello de botella que más movería el retorno del gasto ahí.

**Aviso honesto antes de empezar:** tu documento metodológico menciona 4 posibles cuellos de botella — agua, dispersión, vía de acceso, cobertura de salud. De esas 4, **solo 2 existen tal cual en tus 16 columnas de X** (`pct_agua_permanente` para agua, `densidad_edificios_km2` como proxy de dispersión). **"Vía de acceso" no está en tus datos** — nunca se extrajo una variable de accesibilidad vial en `00_prep_dataset`. **"Cobertura de salud" tampoco está en X** — existe `centro_salud_municipal`, pero es una variable de **W** (gestión municipal), no de X (contexto territorial), así que no se puede usar aquí sin cambiar el diseño del modelo. Se agrega `pct_construido` (grado de urbanización) como tercera variable simulable, a falta de las otras dos. Si necesitas los 4 cuellos de botella originales, hay que volver a `00_prep_dataset.ipynb` a extraer accesibilidad vial y cobertura de salud como X — no se hizo aquí por no tocar ese notebook sin que lo pidas explícitamente.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.dummy import DummyRegressor
from econml.dml import CausalForestDML

cols_X = [
    "pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible",
    "n_edificios", "area_construida_m2", "confianza_media",
    "area_distrito_km2", "densidad_edificios_km2",
    "elevacion_media", "pendiente_media",
    "pct_agua_permanente", "pct_agua_estacional",
    "altitude", "superficie", "pob_densidad_2020",
]

anemia = pd.read_csv("data/predictions/residuos_anemia_model.csv")
spend = pd.read_csv("data/predictions/residuos_spending_model.csv")
df = pd.read_csv("data/clean/merged/causal_model_data.csv")
df = df[df["anio"].between(2021, 2025)].copy()

merged = (
    anemia[["ubigeo", "anio", "residuo_Y"]]
    .merge(spend[["ubigeo", "anio", "residuo_T"]], on=["ubigeo", "anio"], how="inner")
    .merge(df[["ubigeo", "anio", "distrito", "departamento"] + cols_X], on=["ubigeo", "anio"], how="inner")
)
print(f"Filas: {len(merged)}, distritos: {merged['ubigeo'].nunique()}")


## 1. Re-entrenar el mismo Causal Forest

Mismos hiperparámetros y `random_state=42` que en `02_causal_forest.ipynb` — el resultado es idéntico, solo toma ~25 segundos volver a entrenarlo.

In [ ]:
X = merged[cols_X].values
Y_res = merged["residuo_Y"].values
T_res = merged["residuo_T"].values

cf = CausalForestDML(
    model_y=DummyRegressor(strategy="constant", constant=0),
    model_t=DummyRegressor(strategy="constant", constant=0),
    n_estimators=1000,
    min_samples_leaf=10,
    honest=True,
    cv=2,
    random_state=42,
    n_jobs=-1,
)
cf.fit(Y_res, T_res, X=X)
print("Modelo entrenado ✅")


## 2. τ actual por distrito (línea base)

Mismo cálculo que en `02_causal_forest.ipynb`: X promedio por distrito, un τ por distrito.

In [ ]:
dist_X = merged.groupby(["ubigeo", "distrito", "departamento"])[cols_X].mean().reset_index()
Xd = dist_X[cols_X].values
tau_base = cf.effect(Xd)

print(f"τ base — distritos: {len(tau_base)}")
print(pd.Series(tau_base).describe())


## 3. Simulación: una variable a la vez, solo hacia "mejor"

Para cada variable candidata, se sube su valor al **percentil 75 nacional** de esa variable — representa "como si este distrito tuviera el nivel de acceso de un distrito bien servido", no un máximo irreal. Solo se sube si el distrito está POR DEBAJO de ese percentil (`np.maximum`) — nunca se empeora artificialmente a un distrito que ya está mejor que el percentil 75, que sería un error si se asignara el valor fijo sin comparar.

In [ ]:
bottlenecks = {
    "agua_segura": "pct_agua_permanente",
    "dispersion": "densidad_edificios_km2",
    "urbanizacion": "pct_construido",
}
targets = {col: df[col].quantile(0.75) for col in bottlenecks.values()}
print("Valores objetivo (percentil 75 nacional):")
for k, v in bottlenecks.items():
    print(f"  {k} ({v}): {targets[v]:.4f}")

resultados = {"tau_actual": tau_base}
for nombre, col in bottlenecks.items():
    idx = cols_X.index(col)
    X_mod = Xd.copy()
    X_mod[:, idx] = np.maximum(X_mod[:, idx], targets[col])
    tau_mod = cf.effect(X_mod)
    resultados[f"delta_tau_{nombre}"] = tau_mod - tau_base

contra_df = pd.concat(
    [dist_X[["ubigeo", "distrito", "departamento"]], pd.DataFrame(resultados)], axis=1
)
contra_df.head()


## 4. Cuello de botella dominante por distrito

El cuello de botella dominante es la variable cuya mejora simulada produce el mayor `delta_tau`. Si un distrito ya está en el percentil 75 (o mejor) en las 3 variables, no tiene cuello de botella que mostrar con esta metodología — se marca explícitamente en vez de forzar una respuesta.

In [ ]:
delta_cols = [c for c in contra_df.columns if c.startswith("delta_tau_")]

contra_df["mejora_maxima"] = contra_df[delta_cols].max(axis=1)
contra_df["cuello_de_botella_dominante"] = (
    contra_df[delta_cols].idxmax(axis=1).str.replace("delta_tau_", "", regex=False)
)
contra_df.loc[contra_df["mejora_maxima"] <= 1e-6, "cuello_de_botella_dominante"] = (
    "ninguno (ya en percentil 75+)"
)

print("Distribución de cuellos de botella dominantes:")
print(contra_df["cuello_de_botella_dominante"].value_counts())
print()
print("Top 10 distritos con mayor mejora simulada posible:")
contra_df.sort_values("mejora_maxima", ascending=False).head(10)


## 5. Guardar en `data/predictions/`

In [ ]:
OUTPUT_DIR = Path("data/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

contra_df.to_csv(OUTPUT_DIR / "contrafactual_cuellos_de_botella.csv", index=False)
print(f"Guardado: {OUTPUT_DIR / 'contrafactual_cuellos_de_botella.csv'}  ({len(contra_df)} distritos)")
